In [3]:
!scp hfe:/public/home/zongzi/z_works/3_all_flow_jf/works/wf_ins_hfe/9f/dd/85/9fdd85dd-b20c-4601-b7bf-31517d66d2d5_1/jfremote_in.json \
./oth/jfremote_in_big.json

jfremote_in.json                              100% 8742    50.5KB/s   00:00    


In [4]:
!scp hfe:/public/home/zongzi/z_works/3_all_flow_jf/works/testeletrode/ea/44/40/ea4440c8-edc6-4262-8ebe-04cbd17195f6_1/jfremote_in.json \
./oth/jfremote_in_small.json

jfremote_in.json                              100% 8748    39.8KB/s   00:00    


In [ ]:
/public/home/zongzi/z_works/3_all_flow_jf/works/wf_ins_hfe/eb/e3/0b/ebe30b5c-a0f6-4fd3-b4fc-9cc17534b2ff_1

In [2]:
# ========================
# 导入必要模块
# ========================
import warnings
warnings.filterwarnings("ignore")
import numpy as np
#
from pymatgen.ext.matproj import MPRester
from pymatgen.analysis.adsorption import AdsorbateSiteFinder
from pymatgen.core import Element, Molecule, Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import SlabGenerator
#
from atomate2.vasp.jobs.core import RelaxMaker, StaticMaker
from atomate2.vasp.jobs.base import BaseVaspMaker
from atomate2.vasp.flows.core import *
from atomate2.vasp.flows.phonons import PhononMaker
from atomate2.vasp.flows.electrode import ElectrodeInsertionMaker
from atomate2.vasp.powerups import update_user_incar_settings
from atomate2.common.powerups import add_metadata_to_flow
#
from jobflow.utils.graph import to_mermaid
from jobflow import Flow, Job
from jobflow.managers.local import run_locally
from jobflow_remote import submit_flow
from jobflow_remote import JobController
#
import sys
sys.path.append('/Users/jzz/.jfremote')
import os
import shutil
import subprocess
#
# ========================
# 配置管理模块
# ========================
class VASPConfigManager:
    @staticmethod
    def get_default_incar():
        """默认INCAR参数（移除了KPAR和NCORE）"""
        return {
            "ENCUT": 520,
            "PREC": "Normal",
            "EDIFF": 1e-5,
            "EDIFFG": -0.05,
            "ISMEAR": 0,
        }

    @staticmethod
    def get_cluster_config(worker_n):
        """集群资源配置"""
        configs = {
            "hfe": {
                "module": "yaml_generator_hf",
                "resources": {"nodes": 1, "ntasks": 64, "partition": "hfacexclu08"}
            },
            "sc": {
                "module": "yaml_generator_sc_hf",
                "resources": {"nodes": 1, "ntasks": 64, "partition": "dzacexclu01"}
            },
            "cm": {
                "module": "yaml_generator_cm_hf",
                "resources": {"nodes": 1, "ntasks": 64, "partition": "cpu192", "qos": "premium"}
            },
            "gpu": {
                "module": "yaml_generator_gpu_hf",
                "resources": {"nodes": 1, "ntasks_per_node": 16, "partition": "a800", "gres": "gpu:4"}
            },
        }
        return configs.get(worker_n, None)

# ========================
# 工作流构建模块
# ========================
class VASPWorkflowBuilder:
    SPECIAL_JOBS = ['dielectric', 'hse band structure', 'polarization', 'hse static']

    def __init__(self, maker_instance):
        """初始化时传入具体的 Maker 实例"""
        self.maker_instance = maker_instance

    def _get_structure(self, input_data):
        """根据输入路径或材料 ID 加载结构"""
        if input_data.endswith((".cif", ".vasp")):
            return Structure.from_file(input_data)
        elif input_data.startswith("mp-"):
            with MPRester() as mpr:
                return mpr.get_structure_by_material_id(input_data)
        else:
            raise ValueError("输入数据必须是文件路径或材料ID")

    def build_workflow_base(self, input_data, base_incar=None, parr_incar=None):
        """构建基础工作流（例如 Relax, Static, Phonon）"""
        structure = self._get_structure(input_data)
        base_incar = base_incar or VASPConfigManager.get_default_incar()
        parr_incar = parr_incar or {}

        flow = self.maker_instance.make(structure)

        if isinstance(flow, Job):
            flow = Flow([flow])

        return self._update_incar_settings(flow, base_incar, parr_incar)

    def build_workflow_insert(self, input_data, base_incar, parr_incar, inserted_element, n_steps, insertions_per_step):
        """构建电极插层类工作流（用于 ElectrodeInsertionMaker）"""
        structure = self._get_structure(input_data)
        base_incar = base_incar or VASPConfigManager.get_default_incar()
        parr_incar = parr_incar or {}

        # 如果 maker 支持插层参数则使用
        if hasattr(self.maker_instance, "make"):
            flow = self.maker_instance.make(
                structure,
                inserted_element=inserted_element,
                n_steps=n_steps,
                insertions_per_step=insertions_per_step
            )
        else:
            raise ValueError("当前 maker_instance 不支持插层构造")

        if isinstance(flow, Job):
            flow = Flow([flow])

        return self._update_incar_settings(flow, base_incar, parr_incar)

    def _update_incar_settings(self, flow, base_incar, parr_incar):
        """INCAR参数更新逻辑"""
        for job in flow.jobs:
            target_incar = base_incar.copy()
            # 特殊任务不添加 parr_incar 参数
            if job.name not in self.SPECIAL_JOBS:
                target_incar.update(parr_incar)
            
            # 更新 INACR 设置
            flow = update_user_incar_settings(
                flow=flow,
                incar_updates=target_incar,
                name_filter=job.name,
                class_filter=BaseVaspMaker
            )

        return flow

# ========================
# 工作流提交模块
# ========================
class WorkflowSubmitter:
    @staticmethod
    def submit(flow, project=None, worker_n=None):
        """统一提交入口"""
        if project is None:
            return run_locally(flow)
        
        config = VASPConfigManager.get_cluster_config(worker_n)
        if not config:
            raise ValueError(f"不支持的集群配置: {worker_n}")

        try:
            __import__(config["module"])
            generate_yaml = sys.modules[config["module"]].generate_yaml_file
            generate_yaml(project)
            
            return submit_flow(
                flow,
                worker=f"worker_{project}",
                project=project,
                resources=config["resources"]
            )
        except Exception as e:
            print(f"提交错误: {str(e)}")
            raise

# ========================
# 工作流管理模块
# ========================
class WorkflowManager:
    @staticmethod
    def check_states(jc, db_id):
        """
        检查工作流状态
        
        Args:
            jc (JobController): 作业控制器实例
            db_id: 流程数据库ID
            
        Returns:
            dict: 不同状态对应的作业ID字典
        """
        states = {}
        db = db_id
        tem_jc = jc.get_job_doc(db_id=db)
        flow_job = jc.get_flow_info_by_job_uuid(tem_jc.uuid)

        print(jc.get_job_doc(job_id=flow_job['jobs'][0]).job.function_args[0].formula)
        print(flow_job['uuid'], flow_job['state'])
        print('#')

        for job_id in flow_job['jobs']:
            tem_jcc = jc.get_job_doc(job_id)
            state_str = str(tem_jcc.state)
            states.setdefault(state_str, []).append(job_id)

            if state_str not in ['JobState.COMPLETED', 'JobState.WAITING', 'JobState.SUBMITTED']:
                print("#")
                print(tem_jcc.db_id, tem_jcc.state, tem_jcc.job.name)
                print(tem_jcc.run_dir)
                print("#")

        return states

    @staticmethod
    def update_incar(jc, db_id, incar_updates):
        """
        更新失败作业的INCAR参数
        
        Args:
            jc (JobController): 作业控制器实例
            db_id: 作业数据库ID
            incar_updates (dict): 需要更新的INCAR参数
        """
        job_doc = jc.get_job_doc(db_id=db_id)
        print(f"原始 INCAR 设置: {job_doc.job.maker.input_set_generator.user_incar_settings}")

        updated_job = update_user_incar_settings(job_doc.job, incar_updates)
        job_doc.job = updated_job

        jc._set_job_properties(job_doc.as_db_dict(), db_id=db_id)
        updated_job_doc = jc.get_job_doc(db_id=db_id)
        print(f"更新后的 INCAR 设置: {updated_job_doc.job.maker.input_set_generator.user_incar_settings}")

    @staticmethod
    def update_structure(jc, db_id, new_structure):
        """
        更新失败作业的结构参数
        
        Args:
            jc (JobController): 作业控制器实例
            db_id: 作业数据库ID
            new_structure: 新的结构对象
        """
        job_doc = jc.get_job_doc(db_id=db_id)
        if 'structure' in job_doc.job.function_kwargs:
            job_doc.job.function_kwargs['structure'] = new_structure
            jc._set_job_properties(job_doc.as_db_dict(), db_id=db_id)
            print('结构更新成功')
        else:
            print("Function args:", job_doc.job.function_args)
            print("Function kwargs:", job_doc.job.function_kwargs.keys())
            print('需要尝试备份再重置function_args')

    @staticmethod
    def update_resources(jc, db_id, resource_updates):
        """
        更新作业的计算资源配置
        
        Args:
            jc (JobController): 作业控制器实例
            db_id: 作业数据库ID
            resource_updates (dict): 新的资源配置参数
        """
        job_doc = jc.get_job_doc(db_id=db_id)
        if job_doc.resources:
            print(f"原始 resources 设置: {job_doc.resources}")
            job_doc.resources = resource_updates
            jc._set_job_properties(job_doc.as_db_dict(), db_id=db_id)
            updated_job_doc = jc.get_job_doc(db_id=db_id)
            print(f"更新后的 resource 设置: {updated_job_doc.resources}")
        else:
            print('没有找到原始resources')

    @classmethod
    def handle_failures(cls, jc, failed_job_id, project_name, 
                        new_incar=None, new_structure=None, new_resources=None):
        """
        统一处理失败作业
        
        Args:
            jc (JobController): 作业控制器实例
            failed_job_id: 失败作业ID
            project_name (str): 项目名称
            new_incar (dict): 需要更新的INCAR参数
            new_structure: 需要更新的结构对象
            new_resources (dict): 需要更新的资源配置
        """
        db_id = failed_job_id
        tem_jcc = jc.get_job_doc(db_id=db_id)
        
        try:
            if new_incar is not None:
                cls.update_incar(jc, str(db_id), new_incar)
            if new_resources is not None:
                cls.update_resources(jc, str(db_id), new_resources)
            if new_structure is not None:
                cls.update_structure(jc, str(db_id), new_structure)

            jc.rerun_job(db_id=str(db_id), force=True)
        except Exception as e:
            print(f"处理作业 {db_id} 时出错: {e}")

In [3]:
import json
import re
from datetime import datetime
import sys
# sys.path.append('/Users/jzz/jzz_python/z_jupyter/1_jf/z_wf_instance')
# from jf_jzz import (
#     VASPWorkflowBuilder,
#     ElectrodeInsertionMaker,
#     RelaxMaker,
#     StaticMaker,
#     add_metadata_to_flow,
#     WorkflowSubmitter,
#     to_mermaid,
#     BaseVaspMaker,
#     VASPConfigManager,
# )

def run_workflow_and_save_info(project, worker_n, input_data, flow_identifier, base_incar, parr_incar, inserted_element, n_steps, insertions_per_step):
    # 构建工作流
    custom_maker = ElectrodeInsertionMaker(
        relax_maker=RelaxMaker(),
        static_maker=StaticMaker(task_document_kwargs={"store_volumetric_data": []})
    )

    builder = VASPWorkflowBuilder(custom_maker)

    my_flow = builder.build_workflow_insert(
        input_data=input_data,
        base_incar=base_incar,
        parr_incar=parr_incar,
        inserted_element=inserted_element,
        n_steps=n_steps, 
        insertions_per_step=insertions_per_step
    )
    my_flow = add_metadata_to_flow(my_flow, {"flow_identifier": flow_identifier}, class_filter=BaseVaspMaker)

    return my_flow
    # # 提交工作流
    # submitter = WorkflowSubmitter()
    # vis_wf = to_mermaid(my_flow)
    # response = submitter.submit(my_flow, project=project, worker_n=worker_n)

    # # 打印可视化和提交结果
    # print(vis_wf)
    # print(response)

    # # 准备要保存的数据
    # data_to_save = {
    #     "vis_wf": vis_wf,
    #     "project": project,
    #     "response": str(response)
    # }

    # # 获取当前时间的时间戳并格式化
    # current_time = datetime.now().strftime("%Y%m%d%H%M%S")
    # json_file_path = f'output_{current_time}_{str(response[0])}.json'

    # # 将数据保存到 JSON 文件
    # with open(json_file_path, 'w') as f:
    #     json.dump(data_to_save, f, indent=4)

    # print(f"数据已保存到 {json_file_path}")
    # return json_file_path


# if __name__ == "__main__":
#     worker_n = "hfe"
#     project = f"wf_ins_{worker_n}"
#     input_data = "./kvo_old.vasp"
#     flow_identifier = "wf_instance_kvo_electrode"
#     base_incar = VASPConfigManager.get_default_incar()
#     parr_incar = {"KPAR": 4, "NCORE": 16}
#     inserted_element = 'Zn'
#     n_steps = 1
#     insertions_per_step = 3

#     json_file = run_workflow_and_save_info(project, worker_n, input_data, flow_identifier, base_incar, parr_incar, inserted_element, n_steps, insertions_per_step)
#     print(f"生成的 JSON 文件路径: {json_file}")

In [4]:
if __name__ == "__main__":
    worker_n = "hfe"
    project = f"wf_ins_{worker_n}"
    input_data = "./kvo_old.vasp"
    flow_identifier = "wf_instance_kvo_electrode"
    base_incar = VASPConfigManager.get_default_incar()
    parr_incar = {"KPAR": 4, "NCORE": 16}
    inserted_element = 'Zn'
    n_steps = 1
    insertions_per_step = 3

    json_file = run_workflow_and_save_info(project, worker_n, input_data, flow_identifier, base_incar, parr_incar, inserted_element, n_steps, insertions_per_step)

In [5]:
json_file

Flow(name='Flow', uuid='054045bd-4355-4d47-b7d8-2a883709e105')
1. Job(name='relax', uuid='2148750d-3ee2-4284-8d20-63bcdd69b24e')
2. Job(name='get_stable_inserted_results', uuid='1b608dfb-2c54-4cb0-b546-2259dbbb3ea0')
3. Job(name='get_computed_entries', uuid='fb75b860-1d5a-4046-bda5-ebd1fc2ad82a')
4. Job(name='get_structure_group_doc', uuid='2578a7fc-2ab6-4c93-b14d-37dd1cb498cb')

In [18]:
# st = json_file.jobs[1].function_kwargs['static_maker']

# # if issubclass(StaticMaker, BaseVaspMaker):
# #     print("StaticMaker 是 BaseVaspMaker 的子类，会被识别。")

In [6]:
test_flow = json_file

In [12]:
from typing import Dict, Any

def _update_incar_settings(flow, parr_incar: Dict[str, Any]):
    """更新 INCAR 参数（移除冗余的 class_filter，依赖 Maker 自身类型）"""
    # 遍历所有的 jobs，检查是否是子 Flow
    def update_jobs_in_flow(current_flow):
        for job in current_flow.jobs:
            if job.name not in SPECIAL_JOBS:
                incar_updates = {**parr_incar}  # 复制 parr_incar 的内容
                current_flow = update_user_incar_settings(
                    flow=current_flow,
                    incar_updates=incar_updates,
                    name_filter=job.name,  # 确保针对每个 job 更新
                    class_filter=VaspObject
                )
            
            # 如果 job 是子 Flow，递归处理子 Flow 中的 jobs
            if isinstance(job, Flow):
                current_flow = update_jobs_in_flow(job)  # 递归调用

        return current_flow
    
    # 开始更新顶层 flow
    return update_jobs_in_flow(flow)

In [15]:
def update_jobs_in_flow(current_flow):
    for job in current_flow.jobs:
        print(job.name)
        if isinstance(job, Flow):
            current_flow = update_jobs_in_flow(job)

In [24]:
isinstance(test_flow.jobs[1],Flow)

False

In [80]:
test_flow.jobs[1].function_kwargs.keys()

dict_keys(['structure', 'inserted_element', 'structure_matcher', 'static_maker', 'relax_maker', 'get_charge_density', 'n_steps', 'insertions_per_step'])

In [32]:
def print_nested_flows(flow):
    # 遍历当前 Flow 中的 jobs
    for job in flow.jobs:
        if isinstance(job, Job):
            # 打印嵌套 Flow 的名字（如果存在）
            if hasattr(job, 'name'):
                print(f"Found nested Flow with name: {job.name}")
            else:
                print("Found nested Flow without name")
            # 递归调用，检查嵌套的 Flow
            #print_nested_flows(job)

# 调用递归函数来遍历 test_flow
print_nested_flows(test_flow)

Found nested Flow with name: relax
Found nested Flow with name: get_stable_inserted_results
Found nested Flow with name: get_computed_entries
Found nested Flow with name: get_structure_group_doc


In [39]:
# 假设你有一个 Flow，里面包含多个 Job，某些 Job 会动态生成新的 Job
def process_jobs_in_flow(flow):
    # 遍历 flow 中的所有 job
    for job in flow.jobs:
        # 如果 job 执行时生成了新的 Job，更新 flow
        print(f"Processing job: {job.name}")

        # 处理 job 执行后动态生成的任务（即新的 Job）
        if hasattr(job, 'new_jobs') and job.new_jobs:
            print(f"Job {job.name} generated new jobs: {[j.name for j in job.new_jobs]}")
            # 将新生成的 Job 添加到 Flow 中
            flow.jobs.extend(job.new_jobs)

        # 如果 job 还包含其他 Flow（子流），递归处理
        if isinstance(job.output, Flow):
            print(f"Job {job.name} has an output Flow, processing it...")
            process_jobs_in_flow(job.output)  # 递归处理子 Flow

# 假设 test_flow 是你的顶层 Flow
process_jobs_in_flow(test_flow)

Processing job: relax
Processing job: get_stable_inserted_results
Processing job: get_computed_entries
Processing job: get_structure_group_doc


In [57]:
from atomate2.vasp.jobs.base import BaseVaspMaker

def _update_incar_settings(flow: Flow):
    """递归更新所有层级 Job 的 INCAR 参数（包括嵌套 Flow 中的 Job）"""
    def traverse_flow(current_flow: Flow):
        for job_or_flow in current_flow.jobs:  # 遍历当前 Flow 的所有成员（可能是 Job 或子 Flow）
            if isinstance(job_or_flow, Flow):
                # 递归处理子 Flow
                traverse_flow(job_or_flow)
            elif isinstance(job_or_flow, Job):  # 假设 Job 继承自 BaseVaspJob
                print(job_or_flow)    
    traverse_flow(flow)

_update_incar_settings(test_flow)

Job(name='relax', uuid='2148750d-3ee2-4284-8d20-63bcdd69b24e')
Job(name='get_stable_inserted_results', uuid='1b608dfb-2c54-4cb0-b546-2259dbbb3ea0')
Job(name='get_computed_entries', uuid='fb75b860-1d5a-4046-bda5-ebd1fc2ad82a')
Job(name='get_structure_group_doc', uuid='2578a7fc-2ab6-4c93-b14d-37dd1cb498cb')


In [77]:
_update_incar_settings(test_flow)

当前 Job: relax
当前 Job: get_stable_inserted_results
当前 Job: get_computed_entries
当前 Job: get_structure_group_doc


In [58]:
from atomate2.vasp.jobs.base import BaseVaspMaker

def _update_incar_settings(flow: Flow):
    """递归更新所有层级Job的INCAR参数（包括嵌套Flow中的Job）"""
    def traverse_flow(current_flow: Flow):
        for job_or_flow in current_flow.jobs:  # 遍历当前Flow的所有成员（可能是Job或子Flow）
            if isinstance(job_or_flow, Flow):
                print(f"进入子Flow: {job_or_flow.name}")
                # 递归处理子Flow
                traverse_flow(job_or_flow)
            elif isinstance(job_or_flow, Job):  # 假设Job继承自BaseVaspJob
                print(job_or_flow)    
    traverse_flow(flow)

# 假设test_flow是你的测试Flow对象
_update_incar_settings(test_flow)

Job(name='relax', uuid='2148750d-3ee2-4284-8d20-63bcdd69b24e')
Job(name='get_stable_inserted_results', uuid='1b608dfb-2c54-4cb0-b546-2259dbbb3ea0')
Job(name='get_computed_entries', uuid='fb75b860-1d5a-4046-bda5-ebd1fc2ad82a')
Job(name='get_structure_group_doc', uuid='2578a7fc-2ab6-4c93-b14d-37dd1cb498cb')


In [68]:
from jobflow import Flow, Maker, Response, job


def _update_incar_settings(flow):
    """递归遍历所有层级的 Job（包括 Response 包裹的 Flow）"""
    def traverse(node):
        # 处理 Response（atomate2 任务函数的常见返回类型）
        if isinstance(node, Response) and hasattr(node, "replace"):
            traverse(node.replace)  # 解包 Response 中的实际 Flow/Job
        # 处理 Flow（包含 jobs 列表）
        elif isinstance(node, Flow):
            for child in node.jobs:
                traverse(child)  # 递归子节点
        # 处理 Job（叶子节点，假设 Job 有 name 属性）
        elif isinstance(node, Job):
            print(f"找到 Job: {node.name}")  # 验证是否捕获动态 Job
    traverse(flow)

_update_incar_settings(test_flow)

找到 Job: relax
找到 Job: get_stable_inserted_results
找到 Job: get_computed_entries
找到 Job: get_structure_group_doc


In [73]:
from jobflow import Flow, Job, Response

def _update_incar_settings(flow):
    """递归遍历 Jobflow 工作流，包含 Response 替换的嵌套 Flow"""
    def traverse(node):
        # 处理顶层 Flow 或替换的 Flow
        if isinstance(node, Flow):
            for child in node.jobs:
                traverse(child)  # 递归子节点（可能是 Job 或子 Flow）
        
        # 处理 Job（关键：检查是否有替换的 Flow）
        elif isinstance(node, Job):
            print(f"当前 Job: {node.name}")
            
            # 获取 Job 的输出（可能包含 Response 替换的 Flow）
            if hasattr(node, "output"):
                output = node.output
                if isinstance(output, Response) and hasattr(output, "replace"):
                    replace_flow = output.replace
                    if isinstance(replace_flow, Flow):
                        print(f"发现替换 Flow，包含 {len(replace_flow.jobs)} 个 Job")
                        traverse(replace_flow)  # 递归替换的 Flow
                        # 在此处更新替换 Flow 中 Job 的 INCAR
                        for job_in_replace in replace_flow.jobs:
                            print(job_in_replace)
        
        # 处理其他可能的嵌套类型（如列表）
        elif isinstance(node, list):
            for item in node:
                traverse(item)

    traverse(flow)

_update_incar_settings(test_flow)

当前 Job: relax
当前 Job: get_stable_inserted_results
当前 Job: get_computed_entries
当前 Job: get_structure_group_doc


In [76]:
def get_dynamic_jobs_from_flow(flow):
    dynamic_jobs = []

    # 遍历 Flow 中的所有 jobs
    for job in flow.jobs:
        if isinstance(job, Flow):
            # 如果是嵌套的 Flow，递归查找其中的 job
            print(f"Found nested Flow, checking its jobs...")
            dynamic_jobs.extend(get_dynamic_jobs_from_flow(job))  # 递归查找嵌套 Flow 中的 jobs
        else:
            # 如果是 Job 类型，直接添加到结果列表
            dynamic_jobs.append(job)

    return dynamic_jobs

# 假设 test_flow 是你从 get_stable_inserted_results 返回的 Flow
# 在 get_stable_inserted_results 中，jobs 是动态生成的
def get_all_dynamic_jobs(test_flow):
    dynamic_jobs = []
    # 先检查顶层 Flow 的 jobs
    dynamic_jobs.extend(get_dynamic_jobs_from_flow(test_flow))
    
    # 打印所有找到的动态生成的 jobs
    print(f"Found dynamic jobs: {[job.name for job in dynamic_jobs]}")
    return dynamic_jobs

# 示例：假设你已经有了一个 Flow 对象
dynamic_jobs = get_all_dynamic_jobs(test_flow)


Found dynamic jobs: ['relax', 'get_stable_inserted_results', 'get_computed_entries', 'get_structure_group_doc']


In [83]:
def test_tt(flow):
    """更新 INCAR 参数"""
    for job in flow.jobs:
        print(job.name)
test_tt(test_flow)

relax
get_stable_inserted_results
get_computed_entries
get_structure_group_doc


In [1]:
from atomate2.vasp.flows.electrode import ElectrodeInsertionMaker
print(ElectrodeInsertionMaker.__name__)

/Users/jzz/miniconda3/envs/hf_mat/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ElectrodeInsertionMaker


In [8]:
# 'ElectrodeInsertionMaker'==ElectrodeInsertionMaker.__name__